In [3]:
import pandas as pd
import glob
import os

DIR_RAW = '../Data/Raw'
DIR_PROCESSED = '../Data/Processed'

os.makedirs(DIR_PROCESSED, exist_ok=True)

file_list = glob.glob(f"{DIR_RAW}/*.parquet")
print(f"Ditemukan {len(file_list)} file saham mentah yang siap dibersihkan.")

Ditemukan 30 file saham mentah yang siap dibersihkan.


In [4]:
def deteksi_outlier_iqr(dataframe, kolom):
    Q1 = dataframe[kolom].quantile(0.25)
    Q3 = dataframe[kolom].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR
    outlier_mask = (dataframe[kolom] < batas_bawah) | (dataframe[kolom] > batas_atas)
    return outlier_mask

In [5]:
print("Memulai proses data cleaning...\n")

total_processed = 0

for file_path in file_list:
    ticker_name = os.path.basename(file_path).replace('.parquet', '')
    
    df = pd.read_parquet(file_path)
    df_clean = df.ffill().dropna()
    df_clean['Outlier_Close'] = deteksi_outlier_iqr(df_clean, 'Close')
    df_clean['Outlier_Volume'] = deteksi_outlier_iqr(df_clean, 'Volume')
    
    path_simpan = f"{DIR_PROCESSED}/{ticker_name}_clean.parquet"
    df_clean.to_parquet(path_simpan)
    
    total_processed += 1

print(f"Proses selesai! {total_processed} file berhasil dibersihkan dan disimpan ke {DIR_PROCESSED}.")

Memulai proses data cleaning...

Proses selesai! 30 file berhasil dibersihkan dan disimpan ke ../Data/Processed.


In [6]:
if total_processed > 0:
    idx = int(input())
    sample_ticker = os.path.basename(file_list[idx]).replace('.parquet', '')
    sample_path = f"{DIR_PROCESSED}/{sample_ticker}_clean.parquet"
    df_sample = pd.read_parquet(sample_path)
    
    print(f"Preview data bersih untuk saham {sample_ticker}:")
    display(df_sample.head())
    
    print(f"\nStatistik Outlier untuk saham {sample_ticker}:")
    print(f"- Outlier Harga Close : {df_sample['Outlier_Close'].sum()} hari")
    print(f"- Outlier Volume      : {df_sample['Outlier_Volume'].sum()} hari")

Preview data bersih untuk saham META:


,Open,High,Low,Close,Volume,Outlier_Close,Outlier_Volume
Date,,,,,,,
2025-04-30 00:00:00-04:00,536.725593,547.392267,527.853248,547.292603,29244000,False,True
2025-05-01 00:00:00-04:00,590.238618,591.105908,568.725716,570.430420,31159000,False,True
2025-05-02 00:00:00-04:00,581.645443,602.460511,576.531393,595.163269,24739300,False,False
2025-05-05 00:00:00-04:00,589.381237,601.333999,586.221113,597.406250,13887700,False,False
2025-05-06 00:00:00-04:00,590.687195,594.176310,584.755688,585.483398,10600700,False,False



Statistik Outlier untuk saham META:
- Outlier Harga Close : 0 hari
- Outlier Volume      : 17 hari
